# Phase 3: Neural Re-ranking & Evaluation Pipeline
**Team Member 3: Neural Re-ranking & Evaluation Engineer**

This notebook implements the final stage of the CISI Information Retrieval pipeline:
1. **Input**: Top-100 candidate documents from Phase 2 (Hybrid/Dense/BM25 retrieval)
2. **Models**: 
    - Cross-Encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
    - Point-wise Re-ranker (`castorini/monot5-base-msmarco`)
3. **Evaluation**: Compute MRR@10, P@10, NDCG, and Recall using `ranx` library.
4. **Analysis**: Provide detailed query-by-query performance insights.

In [10]:
import sys
import os
import json
import time
import torch
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

# Resolve project root robustly (works from repo root or notebooks/)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

# Add project root to path
sys.path.append(str(PROJECT_ROOT))

# Import project modules
from reranking.cross_encoder_reranker import CrossEncoderReranker
from reranking.monot5_reranker import MonoT5Reranker
from ranx import Qrels, Run, evaluate
from evaluation.eval_pipeline import evaluate_pipeline
from analysis.analyze_reranking_results import generate_report

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: /home/MinhPV/Cisi-retrieval-hybrid-fusion
Using device: cuda
GPU: NVIDIA GeForce RTX 4090


## 1. Load Dataset and Top-100 Candidates
Load the queries, corpus, and the best available top-100 initial retrieval results.

In [11]:
data_dir = Path('data')

with open(data_dir / 'queries.json', 'r') as f:
    queries_data = json.load(f)
    queries = {str(q['query_id']): q['text'] for q in queries_data}

with open(data_dir / 'corpus.json', 'r') as f:
    corpus_data = json.load(f)
    corpus = {str(d['doc_id']): f"{d.get('title', '')} {d.get('text', '')}".strip() for d in corpus_data}

with open(data_dir / 'qrels.json', 'r') as f:
    qrels = json.load(f)

# Fallback mechanism: Try Hybrid -> Dense -> BM25
candidates_file = None
for candidate_name in ['hybrid_top100.json', 'dense_top100.json', 'bm25_top100.json']:
    if (data_dir / candidate_name).exists():
        candidates_file = data_dir / candidate_name
        break

if not candidates_file:
    raise FileNotFoundError("No top-100 candidate files found in data/")

print(f"Loading initial retrieval candidates from: {candidates_file.name}")
with open(candidates_file, 'r') as f:
    top100_candidates = json.load(f)

print(f"Loaded {len(queries)} queries, {len(corpus)} documents.")
print(f"Loaded {len(top100_candidates)} queries with candidates.")

Loading initial retrieval candidates from: dense_top100.json
Loaded 112 queries, 1460 documents.
Loaded 112 queries with candidates.


## 2. Initialize Models
Loading Cross-Encoder and MonoT5 models to GPU.

In [12]:
print("Initializing Cross-Encoder...")
ce_reranker = CrossEncoderReranker()

print("\nInitializing MonoT5...")
monot5_reranker = MonoT5Reranker()

Initializing Cross-Encoder...

Initializing MonoT5...


## 3. Run Neural Re-ranking Pipeline
Re-rank the top-100 candidates using both models. Note: This might take a few minutes depending on GPU.

In [13]:
import time
from tqdm import tqdm

ce_results = {}
monot5_results = {}

ce_total_time = 0.0
t5_total_time = 0.0

for qid in tqdm(top100_candidates.keys(), desc="Re-ranking Queries"):
    query_text = queries[str(qid)]

    # Build candidates in reranker API format: [{doc_id, text, bm25_score}, ...]
    candidates = []
    for d in top100_candidates[qid]:
        doc_id = str(d['doc_id'])
        if doc_id in corpus:
            candidates.append({
                "doc_id": int(doc_id) if doc_id.isdigit() else doc_id,
                "text": corpus[doc_id],
                "bm25_score": float(d.get('score', 0.0)),
            })

    if not candidates:
        continue

    # Cross-Encoder
    start_time = time.time()
    ce_ranked = ce_reranker.rerank(query_text, candidates, batch_size=256)
    ce_total_time += (time.time() - start_time)
    ce_results[str(qid)] = [{"doc_id": d["doc_id"], "score": d["ce_score"]} for d in ce_ranked]

    # MonoT5
    start_time = time.time()
    mt5_ranked = monot5_reranker.rerank(query_text, candidates, batch_size=64)
    t5_total_time += (time.time() - start_time)
    monot5_results[str(qid)] = [{"doc_id": d["doc_id"], "score": d["monot5_score"]} for d in mt5_ranked]

print("\nRe-ranking complete. Saving results...")
with open(data_dir / 'ce_reranked.json', 'w') as f:
    json.dump(ce_results, f, indent=2)
with open(data_dir / 'monot5_reranked.json', 'w') as f:
    json.dump(monot5_results, f, indent=2)

num_queries = max(len(ce_results), 1)
ce_latency_ms = (ce_total_time / num_queries) * 1000
t5_latency_ms = (t5_total_time / num_queries) * 1000

reports_dir = Path('reports')
with open(reports_dir / 'm3_metrics_latency_summary.json', 'w', encoding='utf-8') as f:
    json.dump({
        "candidate_source": candidates_file.name if 'candidates_file' in globals() else None,
        "num_queries": {
            "total_candidates": len(top100_candidates) if 'top100_candidates' in globals() else None,
            "reranked": len(ce_results) if 'ce_results' in globals() else None,
        },
        "latency_ms_per_query": {
            "cross_encoder": float(ce_latency_ms),
            "monot5": float(t5_latency_ms),
        },
        "accuracy": {},  # Will be populated by evaluation pipeline
    }, f, indent=2, ensure_ascii=False)

print("Results and unified JSON summary saved successfully.")
print("\n========= LATENCY SUMMARY =========")
print(pd.DataFrame([
    {"Model": "Cross-Encoder", "Avg Latency/Query (ms)": round(ce_latency_ms, 2)},
    {"Model": "MonoT5", "Avg Latency/Query (ms)": round(t5_latency_ms, 2)},
]).to_markdown(index=False))

Re-ranking Queries: 100%|██████████| 112/112 [00:42<00:00,  2.66it/s]


Re-ranking complete. Saving results...
Results and unified JSON summary saved successfully.

========= LATENCY SUMMARY =========
| Model         |   Avg Latency/Query (ms) |
|:--------------|-------------------------:|
| Cross-Encoder |                   115.69 |
| MonoT5        |                   259.8  |


## 4. Evaluation via RANX 
Computing exact metrics comparing BM25/Dense/Hybrid vs Neural Re-rankers.

In [14]:
# Import the existing evaluation pipeline function to run it and save metrics
print("Running complete evaluation pipeline on all available models...")
evaluate_pipeline()

# Load the unified JSON summary to display metrics
summary_path = Path('reports/m3_metrics_latency_summary.json')
if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    metrics = summary.get('accuracy', {})
    print("\n========= METRICS SUMMARY =========")
    if metrics:
        print(pd.DataFrame(metrics).T.round(4).to_markdown())
    else:
        print("Metrics not yet populated - run evaluation first")
else:
    metrics = {}
    print("Unified summary not found")

# Update the unified JSON with latest metrics if available
if metrics:
    summary['accuracy'] = metrics
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"\nUpdated unified JSON summary: {summary_path}")
else:
    print(f"\nUnified JSON summary: {summary_path} (metrics not updated)")

print(json.dumps(summary, indent=2, ensure_ascii=False))

Running complete evaluation pipeline on all available models...
Loading Qrels...
Loading Neural Re-ranking Results...
Loaded Cross-Encoder results: 112 queries, 11200 total documents
Loaded MonoT5 results: 112 queries, 11200 total documents
Loaded Dense baseline: 112 queries, 11200 total documents
Evaluating neural reranking performance...
Results for Cross-Encoder: {'MRR': 0.5906107918286869, 'P@10': 0.34078947368421053}
Results for MonoT5: {'MRR': 0.4462678760903137, 'P@10': 0.2657894736842106}
Results for Dense-Baseline: {'MRR': 0.61155588439507, 'P@10': 0.39868421052631575}
Updated unified JSON summary: reports/m3_metrics_latency_summary.json
Neural reranking evaluation completed successfully!

========= METRICS SUMMARY =========
|                |    MRR |   P@10 |
|:---------------|-------:|-------:|
| Cross-Encoder  | 0.5906 | 0.3408 |
| MonoT5         | 0.4463 | 0.2658 |
| Dense-Baseline | 0.6116 | 0.3987 |

Updated unified JSON summary: reports/m3_metrics_latency_summary.json


## 5. Detailed Analysis
Leveraging the script `analyze_reranking_results.py` to identify detailed improvements and degradations across all queries.

## 6. Consolidated Single Report
Create one final report file that merges latency + evaluation table + detailed analysis, so you only need to read one report.

In [15]:
# Generate consolidated report from unified JSON summary
summary_path = Path('reports/m3_metrics_latency_summary.json')
analysis_path = Path('reports/phan_tich_xep_hang_lai.md')
final_report_path = Path('reports/m3_final_report.md')

if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)

    # Build latency markdown
    latency_data = summary.get('latency_ms_per_query', {})
    latency_md = "# Reranker Latency Comparison\n\n" + pd.DataFrame([
        {"Model": "Cross-Encoder", "Avg Latency/Query (ms)": round(latency_data.get('cross_encoder', 0), 2)},
        {"Model": "MonoT5", "Avg Latency/Query (ms)": round(latency_data.get('monot5', 0), 2)},
    ]).to_markdown(index=False)

    # Build metrics markdown
    metrics = summary.get('accuracy', {})
    if metrics:
        metrics_md = "# Final Metrics Table\n\n" + pd.DataFrame(metrics).T.round(4).to_markdown()
    else:
        metrics_md = "# Final Metrics Table\n\nNot available."

    # Analysis section
    analysis_md = analysis_path.read_text(encoding='utf-8') if analysis_path.exists() else "# Detailed Analysis\n\nNot available."

    # Combine into final report
    final_md = "\n\n".join([
        "# M3 Final Consolidated Report",
        f"**Candidate Source:** {summary.get('candidate_source', 'Unknown')}",
        f"**Queries Processed:** {summary.get('num_queries', {}).get('reranked', 0)}",
        latency_md,
        metrics_md,
        analysis_md,
    ])

    final_report_path.write_text(final_md, encoding='utf-8')
    print(f"Consolidated report saved: {final_report_path}")
else:
    print("Unified summary not found - cannot generate report")

Consolidated report saved: reports/m3_final_report.md


In [16]:
print("Generating complete analysis report...\n")

# Run the detailed report generation directly
report = generate_report(qrels, queries, top100_candidates, ce_results, monot5_results)

# Display a preview of the report (First 50 lines for brevity)
print("\n--- Report Preview (First 50 lines) ---")
print('\n'.join(report.split('\n')[:50]))

Generating complete analysis report...


--- Report Preview (First 50 lines) ---
# Re-ranking Results Analysis
## Executive Summary
- **Total Queries**: 76
- **Corpus Size**: 1,460 documents
- **Methods Compared**: BM25 (Baseline), Cross-Encoder, MonoT5

## Performance Metrics

| Metric | BM25 | Cross-Encoder | MonoT5 |
|--------|------|---------------|--------|
| MRR@10 | 0.4123 | 0.3967 | 0.3003 |
| MRR@100 | 0.4150 | 0.4008 | 0.3055 |
| Recall@10 | 0.1330 | 0.1286 | 0.0949 |
| Recall@100 | 0.4706 | 0.4706 | 0.4706 |
| NDCG@10 | 0.4229 | 0.3766 | 0.2940 |
| NDCG@100 | 0.4089 | 0.3944 | 0.3632 |

## First Relevant Document Analysis

### BM25
- **Queries with relevant doc in top 100**: 75
- **Queries with no relevant doc**: 1
- **Median Rank of First Relevant Doc**: 2.0
- **Mean Rank of First Relevant Doc**: 6.17
- **Max Rank of First Relevant Doc**: 79

### Cross-Encoder
- **Queries with relevant doc in top 100**: 75
- **Queries with no relevant doc**: 1
- **Median Rank of First Relev